# Chassis frame

The spaceframe as a beam finite-element model (`lapsim/subsystems/frame.py`, tubes from
`cars/ct17_ic/frame_tubes.csv`): mass, centre of gravity and torsional stiffness from a twist test on the
suspension pick-ups. The frame enters the lap simulation two ways: its mass and CG move the car's, and its
stiffness sits between the front and rear roll stiffness, so a soft frame weakens what springs and bars do
to the balance. The website's frame editor uses the same model.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
F = lapsim.subsystems.frame
CAR = lapsim.load_car("ct17_ic")
TUBES = F.to_car(F.load_tubes(CAR.source.parent / "frame_tubes.csv"))
PICK = F.load_pickups(CAR.vehicle.front.hardpoint_file, CAR.vehicle.rear.hardpoint_file)
wall = W.FloatSlider(value=1.0, min=0.6, max=1.6, step=0.05, description="wall thickness x", continuous_update=False)
mat = W.Dropdown(options={"4130 steel": (205e9, 80e9, 7850.0), "6061-T6 aluminium": (68.9e9, 26e9, 2700.0),
                          "Ti-3Al-2.5V": (100e9, 40e9, 4480.0)}, description="material")
use = W.Checkbox(value=True, description="frame twist in the lap simulation")
out, out_lap = W.Output(), W.Output()


def props():
    import dataclasses as dc
    E, G, rho = mat.value
    tubes = [dc.replace(t, wall=min(t.wall * wall.value, t.od / 2)) for t in TUBES]
    m = np.array([rho * t.area * t.length for t in tubes])
    mid = np.array([0.5 * (t.p0 + t.p1) for t in tubes])
    fm = F.FrameModel(tubes, PICK, {"E": E, "G": G, "density": rho})
    tt = fm.torsion_test()
    return {"mass": float(m.sum()), "cg": (m[:, None] * mid).sum(0) / m.sum(), "k": float(tt["stiffness"]),
            "k_deg": float(tt["stiffness_per_deg"])}


BASE = None


def draw(*_):
    global BASE
    with out:
        out.clear_output(wait=True)
        p = props()
        if BASE is None:
            BASE = p
        print(f"frame mass {p['mass']:.1f} kg (baseline {BASE['mass']:.1f}),  torsional stiffness {p['k_deg']:.0f} N m/deg "
              f"(baseline {BASE['k_deg']:.0f}),  CG height {1e3 * p['cg'][2]:.0f} mm")


def laps():
    p = props()
    c = CAR.copy()
    c.vehicle = F.swap_frame(c.vehicle, BASE, p)
    if use.value:
        c.vehicle.mass.chassis_torsional_stiffness = p["k"]
    display(c.run(("skidpad", "autocross")).table().round(3))


for w_ in (wall, mat):
    w_.observe(draw, "value")
draw()
display(W.VBox([wall, mat, use, out, run_button("Run skidpad + autocross", laps, out_lap), out_lap]))